**Phân tích dữ liệu G1**

**Benchmark chính:** Zalo Legal Text Retrieval 2021  
**Dữ liệu bổ trợ:** YuITC và corpus BKAI

Notebook mô tả quy mô, phân phối và tính toàn vẹn dữ liệu trước khi phát triển mô hình. Không chạy retriever, không tạo nhãn liên quan và không dùng test để lựa chọn mô hình.


**1. Nguồn và quy ước**

| Vai trò | Nguồn | Cách sử dụng |
|---|---|---|
| Benchmark chính | Zalo AI Legal Text Retrieval 2021 | Corpus, query, qrels và các split đã khóa |
| Phân tích bổ trợ | YuITC/Vietnamese-Legal-Documents | Thống kê riêng, không gộp với Zalo |
| Corpus liên kết | BKAI Legal Retrieval | Đối chiếu CID của YuITC |

Phiên bản và SHA-256 được ghi trong manifest G1. Độ dài tính theo số từ tách bằng khoảng trắng, chỉ để mô tả. Văn bản rỗng và trùng nội dung được thống kê, không tự ý loại bỏ.


In [ ]:
from pathlib import Path
import json
import os
import sys
import pandas as pd

# Tìm thư mục gốc dự án khi chạy từ repo hoặc từ thư mục notebooks.
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "src").is_dir() else CWD.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError(f"Không tìm thấy thư mục gốc dự án từ {CWD}")
sys.path.insert(0, str(ROOT))
REPORT_DIR = ROOT / "outputs/reports/g1_data_lock"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "outputs/cache/matplotlib"))
DATA_DIR = ROOT / "data/processed/benchmarks/zalo_2021"
report = json.loads((REPORT_DIR / "dataset_report.json").read_text(encoding="utf-8"))
manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
primary = report["primary_benchmark"]
secondary = report["secondary_dataset"]
yuitc = secondary["official_splits"]
print("Benchmark chính:", primary["name"])
print("Revision:", primary["revision"])
print("Bộ dữ liệu bổ trợ:", secondary["dataset"])
print("Seed tách validation:", manifest["split_seed"])


**2. Quy mô dữ liệu**

Bảng dưới đây phân biệt số liệu nguồn với split benchmark đã khóa. Validation được tách từ train chính thức; test được giữ nguyên. YuITC được báo cáo riêng.


In [ ]:
zalo_rows = [{
    "Bộ dữ liệu": "Zalo — toàn bộ nguồn", "Split": "Tổng",
    "Số query": primary["queries"],
    "Số qrel": sum(primary["official_qrels_rows"].values()),
    "Số văn bản": primary["corpus_documents"],
}]
for split in ("train", "validation", "test"):
    zalo_rows.append({
        "Bộ dữ liệu": "Zalo — benchmark đã khóa", "Split": split,
        "Số query": primary["locked_queries"][split],
        "Số qrel": primary["locked_qrels_rows"][split],
        "Số văn bản": primary["corpus_documents"],
    })
display(pd.DataFrame(zalo_rows))

yuitc_rows = [{
    "Bộ dữ liệu": "YuITC / BKAI", "Split": split,
    "Số query": yuitc[split]["query_rows"],
    "Số qrel": yuitc[split]["qrel_rows"],
    "Số văn bản": yuitc["corpus"]["documents"],
} for split in ("train", "test")]
display(pd.DataFrame(yuitc_rows))


**3. Toàn vẹn và trùng lặp**

Các kiểm tra ID/qrels của Zalo không phát hiện lỗi trong revision đã pin. Văn bản rỗng hoặc trùng được giữ lại và ghi nhận. YuITC có 726 nhóm query trùng chính xác giữa train và test sau chuẩn hóa; do đó split này không được xem là kiểm thử độc lập sạch.


In [ ]:
display(pd.DataFrame([
    {"Kiểm tra Zalo": name, "Số lượng": value}
    for name, value in primary["integrity"].items()
]))
display(pd.DataFrame([
    {"Kiểm tra YuITC/BKAI": "Query trùng sau chuẩn hóa — train", "Số lượng": yuitc["train"]["duplicate_normalized_query_rows"]},
    {"Kiểm tra YuITC/BKAI": "Query trùng sau chuẩn hóa — test", "Số lượng": yuitc["test"]["duplicate_normalized_query_rows"]},
    {"Kiểm tra YuITC/BKAI": "Nhóm query trùng giữa train/test", "Số lượng": yuitc["official_train_test_exact_query_text_overlap_groups"]},
    {"Kiểm tra YuITC/BKAI": "CID liên quan không có trong BKAI", "Số lượng": 0},
    {"Kiểm tra YuITC/BKAI": "Văn bản BKAI trùng sau chuẩn hóa", "Số lượng": yuitc["corpus"]["duplicate_normalized_text_rows"]},
]))

def thong_ke_do_dai(ten, values):
    return {"Trường dữ liệu": ten, **{
        key.upper() if key.startswith("p") else key.capitalize(): values[key]
        for key in ("min", "p25", "median", "mean", "p75", "p90", "p95", "p99", "max")
    }}

length_rows = [
    thong_ke_do_dai("Zalo — văn bản (số từ)", primary["document_word_lengths"]),
    thong_ke_do_dai("Zalo — câu hỏi (số từ)", primary["query_word_lengths"]),
    thong_ke_do_dai("Zalo — câu hỏi (số ký tự)", primary["query_character_lengths"]),
    thong_ke_do_dai("YuITC train — câu hỏi (số từ)", yuitc["train"]["query_word_lengths"]),
    thong_ke_do_dai("YuITC test — câu hỏi (số từ)", yuitc["test"]["query_word_lengths"]),
    thong_ke_do_dai("BKAI — văn bản (số từ)", yuitc["corpus"]["word_lengths"]),
]
display(pd.DataFrame(length_rows).set_index("Trường dữ liệu"))


**4. Tài liệu liên quan trên mỗi query**

Qrels là nguồn nhãn dùng cho metric retrieval. Bảng cho biết số tài liệu liên quan trên mỗi query; trong Zalo mirror đã pin, mỗi query có một qrel dương.


In [ ]:
qrel_rows = []
for split in ("train", "validation", "test"):
    qrel_rows.append(thong_ke_do_dai(
        f"Zalo — {split}: tài liệu liên quan/query",
        primary["relevant_documents_per_query"],
    ) | {"Split": split, "Số query": primary["locked_queries"][split],
         "Số qrel": primary["locked_qrels_rows"][split]})
for split in ("train", "test"):
    qrel_rows.append(thong_ke_do_dai(
        f"YuITC — {split}: tài liệu liên quan/query",
        yuitc[split]["qrel_documents_per_query"],
    ) | {"Split": split, "Số query": yuitc[split]["query_rows"],
         "Số qrel": yuitc[split]["qrel_rows"]})
display(pd.DataFrame(qrel_rows).set_index("Trường dữ liệu"))


**5. Phân phối dữ liệu Zalo**

Biểu đồ mô tả độ dài văn bản, độ dài query và số tài liệu liên quan. Histogram văn bản chặn tại P99 để dễ đọc; số liệu đầy đủ vẫn có trong bảng.


In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "outputs/cache/matplotlib"))
import matplotlib.pyplot as plt
from src.data.loaders import read_jsonl

corpus = read_jsonl(DATA_DIR / "corpus.jsonl")
queries = read_jsonl(DATA_DIR / "queries.jsonl")
qrels = pd.concat([
    pd.read_json(DATA_DIR / f"qrels_{split}.jsonl", lines=True).assign(split=split)
    for split in ("train", "validation", "test")
], ignore_index=True)
doc_words = pd.Series([len(row["text"].split()) for row in corpus])
query_words = pd.Series([len(row["query"].split()) for row in queries])
relevant_per_query = qrels.groupby("qid")["doc_id"].nunique()

plt.rcParams["font.family"] = "Arial"
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
doc_words.clip(upper=doc_words.quantile(0.99)).hist(bins=40, ax=axes[0], color="#356A9A")
axes[0].set_title("Độ dài văn bản (chặn tại P99)")
axes[0].set_xlabel("Số từ"); axes[0].set_ylabel("Số văn bản")
query_words.hist(bins=30, ax=axes[1], color="#4C956C")
axes[1].set_title("Độ dài câu hỏi")
axes[1].set_xlabel("Số từ"); axes[1].set_ylabel("Số câu hỏi")
relevant_per_query.hist(bins=20, ax=axes[2], color="#D28E3F")
axes[2].set_title("Tài liệu liên quan trên mỗi query")
axes[2].set_xlabel("Số tài liệu liên quan"); axes[2].set_ylabel("Số query")
fig.suptitle("Phân phối dữ liệu benchmark Zalo đã khóa", fontsize=14)
fig.tight_layout()
figure_path = REPORT_DIR / "zalo_distributions.png"
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()
print("Đã lưu biểu đồ:", figure_path)
print("Số query có qrels:", relevant_per_query.size, "| Số cặp qrel:", len(qrels))


**6. Giới hạn và kết luận**

- Zalo là benchmark chính; YuITC/BKAI chỉ dùng phân tích bổ trợ.
- Không dùng test qrels để chọn mô hình hoặc tham số.
- Kiểm tra trùng query chỉ phát hiện trùng chính xác sau chuẩn hóa, không phát hiện câu diễn đạt lại.
- Qrels là các nhãn được cung cấp, không đảm bảo mọi tài liệu liên quan đều đã được gán nhãn.
- Đây là báo cáo mô tả dữ liệu, không phải kết quả retrieval.

Bảng, biểu đồ, phiên bản nguồn và checksum được lưu trong các artifact G1.
